<a href="https://colab.research.google.com/github/AthaBPratama/Tugas-BIG-DAYTA/blob/main/Dashboard_Big_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
import os

# Cek apakah file dataset_final.csv sudah diunggah
if os.path.exists("dataset_final.csv"):
    print("✓ Berhasil mendeteksi file dataset_final.csv asli Anda.")
    df = pd.read_csv("dataset_final.csv")
else:
    print("⚠ File dataset_final.csv tidak ditemukan. Membuat data simulasi (dummy) agar kode tidak error...")
    # Membuat data dummy untuk simulasi
    np.random.seed(42)
    brands = ['ASUS', 'Lenovo', 'HP', 'Acer', 'Apple']
    data = {
        'Brand': np.random.choice(brands, 100),
        'RAM': np.random.choice([4, 8, 16, 32], 100, p=[0.2, 0.4, 0.3, 0.1]),
        'SSD': np.random.choice([256, 512, 1024], 100, p=[0.3, 0.5, 0.2]),
        'Processor': np.random.choice(['Intel i3', 'Intel i5', 'Intel i7', 'Ryzen 5', 'Ryzen 7'], 100)
    }
    # Rumus logis buatan untuk harga biar korelasinya masuk akal
    data['Price'] = 3000000 + (data['RAM'] * 400000) + (data['SSD'] * 5000) + np.random.normal(0, 500000, 100)
    df = pd.DataFrame(data)

# Tampilkan 5 data teratas
df.head()

⚠ File dataset_final.csv tidak ditemukan. Membuat data simulasi (dummy) agar kode tidak error...


,Brand,RAM,SSD,Processor,Price
0,Acer,32,512,Ryzen 5,1.860061e+07
1,Apple,16,512,Intel i7,1.219353e+07
2,HP,8,256,Ryzen 5,8.048509e+06
3,Apple,4,512,Intel i5,7.383445e+06
4,Apple,8,512,Intel i5,8.187526e+06


In [12]:
print("======== RINGKASAN DATA LAPTOP ========")
print(f"Total Laptop Terdata : {len(df)} unit")
print(f"Rata-rata Harga      : Rp {df['Price'].mean():,.0f}")
print(f"Harga Tertinggi      : Rp {df['Price'].max():,.0f}")
print(f"Harga Terendah       : Rp {df['Price'].min():,.0f}")
print(f"Rata-rata Ukuran RAM : {df['RAM'].mean():.1f} GB")
print("=======================================")

# Menampilkan informasi tipe data kolom
df.info()

======== RINGKASAN DATA LAPTOP ========
Total Laptop Terdata : 100 unit
Rata-rata Harga      : Rp 10,312,112
Harga Tertinggi      : Rp 21,478,447
Harga Terendah       : Rp 5,688,256
Rata-rata Ukuran RAM : 11.5 GB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Brand      100 non-null    object 
 1   RAM        100 non-null    int64  
 2   SSD        100 non-null    int64  
 3   Processor  100 non-null    object 
 4   Price      100 non-null    float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ KB


In [13]:
# Grafik 1: Rata-rata Harga Berdasarkan Brand
df_brand = df.groupby('Brand')['Price'].mean().reset_index().sort_values(by='Price', ascending=False)
fig1 = px.bar(df_brand, x='Brand', y='Price',
             title='Rata-rata Harga Laptop Berdasarkan Brand',
             labels={'Price': 'Rata-rata Harga (Rp)'},
             color='Brand', template='plotly_dark')
fig1.show()

# Grafik 2: Distribusi RAM di Pasar
fig2 = px.pie(df, names='RAM', title='Persentase Distribusi Kapasitas RAM Laptop',
             template='plotly_dark', hole=0.3)
fig2.show()

In [14]:
# Bersihkan data dari nilai kosong jika ada
df_clean = df.dropna(subset=['RAM', 'SSD', 'Price'])

# Menentukan variabel independen (X) dan dependen (y)
X = df_clean[['RAM', 'SSD']]
y = df_clean['Price']

# Inisialisasi dan latih model
model = LinearRegression()
model.fit(X, y)

# Mengecek skor akurasi (R-squared)
score = model.score(X, y)
print(f"Model berhasil dilatih dengan nilai R² (Akurasi Relatif): {score:.2f}")

# Visualisasi Hasil Prediksi vs Harga Asli
df_clean['Prediksi_Harga'] = model.predict(X)
fig3 = px.scatter(df_clean, x='Price', y='Prediksi_Harga', color='Brand',
                 trendline="ols", title="Perbandingan Harga Asli vs Hasil Prediksi Model",
                 labels={'Price': 'Harga Aktual (Rp)', 'Prediksi_Harga': 'Harga Prediksi (Rp)'},
                 template='plotly_dark')
fig3.show()

Model berhasil dilatih dengan nilai R² (Akurasi Relatif): 0.97


In [15]:
#@title 🎛️ Simulator Prediksi Harga Laptop Baru (Geser Parameter)
#@markdown Pilih spesifikasi di bawah ini untuk mengestimasi harga jual pasarannya:

Input_RAM_GB = 16 #@param {type:"slider", min:4, max:64, step:4}
Input_SSD_GB = 2048 #@param {type:"slider", min:128, max:2048, step:128}

# Melakukan prediksi berdasarkan input form
harga_prediksi = model.predict([[Input_RAM_GB, Input_SSD_GB]])[0]

print("==================================================")
print(f" HASIL ESTIMASI UNTUK SPESIFIKASI:")
print(f" 💾 RAM : {Input_RAM_GB} GB")
print(f" 💽 SSD : {Input_SSD_GB} GB")
print("--------------------------------------------------")
print(f" 💰 Perkiraan Harga Jual: Rp {max(0, harga_prediksi):,.0f}")
print("==================================================")

 HASIL ESTIMASI UNTUK SPESIFIKASI:
 💾 RAM : 16 GB
 💽 SSD : 2048 GB
--------------------------------------------------
 💰 Perkiraan Harga Jual: Rp 19,574,090


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LinearRegression was fitted with feature names

